# LeetCode 3749 — Evaluate Valid Expressions bằng Python

- Link: https://leetcode.com/problems/evaluate-valid-expressions/
- Độ khó: **Hard**
- Chủ đề: Phân tích cú pháp (parsing), đệ quy xuống (recursive descent), **ngăn xếp**, ký pháp Ba Lan (prefix notation)

> **Ghi chú:** Đây là bài **Premium**. Đề bài và skeleton lấy từ bản doocs (leetcode.doocs.org); tên hàm `evaluateExpression` theo lời giải tham khảo ở đó — nên đối chiếu lại với skeleton thật nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def evaluateExpression(self, expression: str) -> int:
```

## 1. Mục tiêu học

1. Đọc ngữ pháp `E := số nguyên | op(E,E)` và viết **bộ phân tích đệ quy xuống**.
2. Nhận ra giới hạn **độ sâu đệ quy** của Python (mặc định 1000) với chuỗi dài `10^5` — và chuyển sang **ngăn xếp**.
3. Phân biệt dấu `-` của **số âm** với phép `sub`.
4. Thấy rằng dấu ngoặc là **thừa**: vì mọi phép toán có đúng 2 đối số, dãy token là ký pháp Ba Lan (prefix) ⇒ tính từ phải sang trái bằng một ngăn xếp.
5. Dùng phép chia số nguyên chính xác (không dùng `float`).

## 2. Đề bài & ví dụ

Biểu thức hợp lệ là một **số nguyên** (có thể âm) hoặc có dạng `op(a,b)` với `op ∈ {add, sub, mul, div}` và `a`, `b` là biểu thức hợp lệ. `add = a + b`, `sub = a − b`, `mul = a · b`, `div = a / b` (đề bảo đảm luôn chia hết). Trả về giá trị biểu thức.

**Ví dụ 1:** `"add(2,3)"` → `5`.

**Ví dụ 2:** `"-42"` → `-42`.

**Ví dụ 3:** `"div(mul(4,sub(9,5)),add(1,1))"` → `8`.

**Ràng buộc:** độ dài `≤ 10^5`; mọi kết quả trung gian vừa kiểu `long`; mọi phép chia đều chia hết.

In [1]:
import re, random, sys, time

examples = [("add(2,3)", 5), ("-42", -42), ("div(mul(4,sub(9,5)),add(1,1))", 8)]
for e, exp in examples:
    print(e, "-> kỳ vọng", exp)

add(2,3) -> kỳ vọng 5
-42 -> kỳ vọng -42
div(mul(4,sub(9,5)),add(1,1)) -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Tách token

Các token: tên phép toán (`add`, `sub`, `mul`, `div`), số nguyên (có thể có `-` đứng đầu), `(`, `,`, `)`. Một biểu thức chính quy là đủ.

In [2]:
TOKEN = re.compile(r"add|sub|mul|div|-?\d+|[(),]")
print(TOKEN.findall("div(mul(4,sub(9,-5)),add(1,1))"))

['div', '(', 'mul', '(', '4', ',', 'sub', '(', '9', ',', '-5', ')', ')', ',', 'add', '(', '1', ',', '1', ')', ')']


### 3.2. Dấu `-` không gây nhập nhằng

Phép trừ được viết bằng chữ `sub`, nên ký tự `-` **chỉ** có thể là dấu của số âm. Sau `(` hoặc `,` (hoặc đầu chuỗi), nếu gặp `-` hoặc chữ số thì đó là một số.

### 3.3. Giới hạn đệ quy trong Python

In [3]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
deep = "add(" * 3000 + "1" + ",1)" * 3000
print("độ dài biểu thức lồng 3000 tầng:", len(deep))

giới hạn đệ quy mặc định: 1000
độ dài biểu thức lồng 3000 tầng: 21001


### 3.4. Phép chia số nguyên chính xác

`a // b` trong Python làm tròn **xuống**; vì đề bảo đảm chia hết nên `a // b` cho đúng thương (kể cả số âm). Không dùng `int(a / b)` — `float` chỉ có 53 bit, sai với số cỡ `10^18`.

In [4]:
a, b = 9007199254740993 * 3, 3
print("a // b =", a // b, "| int(a / b) =", int(a / b))

a // b = 9007199254740993 | int(a / b) = 9007199254740992


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cấu trúc biểu thức là một **cây nhị phân**: lá là số, nút trong là phép toán. Đánh giá = duyệt hậu thứ tự (tính hai con rồi áp dụng phép toán).

- Đệ quy: `parse(i)` đọc một biểu thức bắt đầu tại `i`, trả về `(giá trị, vị trí tiếp theo)`.
- Ngăn xếp: gặp tên phép toán thì đẩy vào ngăn xếp phép toán; gặp số thì đẩy vào ngăn xếp giá trị; gặp `)` thì lấy hai giá trị và một phép toán ra, tính, đẩy kết quả lại.

## 5. Cách 1 — Brute Force: thay thế biểu thức trong cùng lặp lại

Tìm mẫu `op(số,số)` (biểu thức không còn lồng) bằng regex, thay bằng giá trị, lặp tới khi chỉ còn một số.

In [5]:
INNER = re.compile(r"(add|sub|mul|div)\((-?\d+),(-?\d+)\)")

def apply(op, a, b):
    if op == "add": return a + b
    if op == "sub": return a - b
    if op == "mul": return a * b
    return a // b

def brute(expr):
    while True:
        new = INNER.sub(lambda m: str(apply(m.group(1), int(m.group(2)), int(m.group(3)))), expr)
        if new == expr:
            return int(expr)
        expr = new

for e, exp in examples:
    print(brute(e), "kỳ vọng", exp)

5 kỳ vọng 5
-42 kỳ vọng -42
8 kỳ vọng 8


**Nút thắt:** mỗi vòng quét lại cả chuỗi, số vòng = độ sâu cây ⇒ `O(n · depth)`, tới `O(n²)` với biểu thức lồng sâu.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát then chốt:** mỗi phép toán có **đúng hai** đối số ⇒ bỏ hết `(`, `,`, `)` ta được dãy token ở **ký pháp Ba Lan** (tiền tố), vẫn xác định duy nhất cây. Tính ký pháp Ba Lan: duyệt **từ phải sang trái**, số thì đẩy vào ngăn xếp, phép toán thì lấy `a` (đỉnh) và `b` (kế tiếp) — đúng thứ tự trái/phải — rồi đẩy `op(a, b)`.

Cả cách ngăn xếp hai chiều (theo dấu ngoặc) và cách Ba Lan đều `O(n)`, không đệ quy.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy xuống (recursive descent) — `O(n)` nhưng có thể `RecursionError` khi lồng sâu.
- **Cách 3:** hai ngăn xếp (phép toán, giá trị) theo dấu ngoặc — `O(n)` (lời giải nộp).
- **Cách 4:** bỏ ngoặc ⇒ ký pháp Ba Lan, tính từ phải sang trái — `O(n)`.

### 7.1. Cách 2 — Đệ quy xuống

In [6]:
def recursive(expr):
    def parse(i):
        if expr[i] == "-" or expr[i].isdigit():
            j = i + 1
            while j < len(expr) and expr[j].isdigit():
                j += 1
            return int(expr[i:j]), j
        op = expr[i:i + 3]
        a, j = parse(i + 4)                 # bỏ qua "op("
        b, j = parse(j + 1)                 # bỏ qua ","
        return apply(op, a, b), j + 1       # bỏ qua ")"
    return parse(0)[0]

for e, exp in examples:
    print(recursive(e), "kỳ vọng", exp)
try:
    recursive(deep)
except RecursionError as err:
    print("biểu thức lồng 3000 tầng ->", type(err).__name__)

5 kỳ vọng 5
-42 kỳ vọng -42
8 kỳ vọng 8
biểu thức lồng 3000 tầng -> RecursionError


### 7.2. Cách 3 — Hai ngăn xếp theo dấu ngoặc (lời giải chính)

In [7]:
def two_stacks(expr):
    ops, vals = [], []
    i, n = 0, len(expr)
    while i < n:
        ch = expr[i]
        if ch == "-" or ch.isdigit():
            j = i + 1
            while j < n and expr[j].isdigit():
                j += 1
            vals.append(int(expr[i:j])); i = j
        elif ch.isalpha():
            ops.append(expr[i:i + 3]); i += 4          # tên phép toán + "("
        elif ch == ")":
            b = vals.pop(); a = vals.pop()
            vals.append(apply(ops.pop(), a, b)); i += 1
        else:                                           # ","
            i += 1
    return vals[0]

for e, exp in examples:
    print(two_stacks(e), "kỳ vọng", exp)
print("lồng 3000 tầng:", two_stacks(deep))

5 kỳ vọng 5
-42 kỳ vọng -42
8 kỳ vọng 8
lồng 3000 tầng: 3001


### 7.3. Cách 4 — Ký pháp Ba Lan, tính từ phải sang trái

In [8]:
def polish(expr):
    tokens = [t for t in TOKEN.findall(expr) if t not in "(),"]
    st = []
    for t in reversed(tokens):
        if t in ("add", "sub", "mul", "div"):
            a = st.pop(); b = st.pop()                 # a là đối số TRÁI
            st.append(apply(t, a, b))
        else:
            st.append(int(t))
    return st[0]

for e, exp in examples:
    print(polish(e), "kỳ vọng", exp)
print("dãy Ba Lan của ví dụ 3:", [t for t in TOKEN.findall(examples[2][0]) if t not in "(),"])

5 kỳ vọng 5
-42 kỳ vọng -42
8 kỳ vọng 8
dãy Ba Lan của ví dụ 3: ['div', 'mul', '4', 'sub', '9', '5', 'add', '1', '1']


## 8. TRACE từng bước

In [9]:
def trace_two_stacks(expr):
    ops, vals = [], []
    i, n = 0, len(expr)
    while i < n:
        ch = expr[i]
        if ch == "-" or ch.isdigit():
            j = i + 1
            while j < n and expr[j].isdigit(): j += 1
            vals.append(int(expr[i:j])); print(f"số {expr[i:j]:>4}: vals = {vals}"); i = j
        elif ch.isalpha():
            ops.append(expr[i:i + 3]); print(f"phép {expr[i:i+3]}: ops = {ops}"); i += 4
        elif ch == ")":
            b = vals.pop(); a = vals.pop(); op = ops.pop(); r = apply(op, a, b)
            vals.append(r); print(f"')': {op}({a},{b}) = {r} -> vals = {vals}, ops = {ops}"); i += 1
        else:
            i += 1
    return vals[0]

trace_two_stacks("div(mul(4,sub(9,5)),add(1,1))")

phép div: ops = ['div']
phép mul: ops = ['div', 'mul']
số    4: vals = [4]
phép sub: ops = ['div', 'mul', 'sub']
số    9: vals = [4, 9]
số    5: vals = [4, 9, 5]
')': sub(9,5) = 4 -> vals = [4, 4], ops = ['div', 'mul']
')': mul(4,4) = 16 -> vals = [16], ops = ['div']
phép add: ops = ['div', 'add']
số    1: vals = [16, 1]
số    1: vals = [16, 1, 1]
')': add(1,1) = 2 -> vals = [16, 2], ops = ['div']
')': div(16,2) = 8 -> vals = [8], ops = []


8

In [10]:
def trace_polish(expr):
    tokens = [t for t in TOKEN.findall(expr) if t not in "(),"]
    print("token Ba Lan:", tokens)
    st = []
    for t in reversed(tokens):
        if t in ("add", "sub", "mul", "div"):
            a = st.pop(); b = st.pop(); st.append(apply(t, a, b))
            print(f"  {t}({a},{b}) -> stack {st}")
        else:
            st.append(int(t)); print(f"  số {t} -> stack {st}")
    return st[0]

trace_polish("sub(-3,mul(2,-4))")

token Ba Lan: ['sub', '-3', 'mul', '2', '-4']
  số -4 -> stack [-4]
  số 2 -> stack [-4, 2]
  mul(2,-4) -> stack [-8]
  số -3 -> stack [-8, -3]
  sub(-3,-8) -> stack [5]


5

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def evaluateExpression(self, expression: str) -> int:
        ops, vals = [], []
        s = expression
        i, n = 0, len(s)
        while i < n:
            ch = s[i]
            if ch == '-' or ch.isdigit():
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                vals.append(int(s[i:j]))
                i = j
            elif ch.isalpha():
                ops.append(s[i:i + 3])
                i += 4
            elif ch == ')':
                b = vals.pop()
                a = vals.pop()
                op = ops.pop()
                if op == 'add':
                    vals.append(a + b)
                elif op == 'sub':
                    vals.append(a - b)
                elif op == 'mul':
                    vals.append(a * b)
                else:
                    vals.append(a // b)        # đề bảo đảm chia hết
                i += 1
            else:
                i += 1
        return vals[0]

sol = Solution()
for e, exp in examples:
    got = sol.evaluateExpression(e)
    print(e, "->", got, "✔" if got == exp else "✘")

add(2,3) -> 5 ✔
-42 -> -42 ✔
div(mul(4,sub(9,5)),add(1,1)) -> 8 ✔


## 10. Vì sao thuật toán đúng

**Bất biến (Cách 3):** khi đọc xong một biểu thức con hoàn chỉnh `E` bắt đầu tại vị trí `p`, ngăn xếp `vals` có thêm đúng một phần tử là giá trị của `E`, và `ops` trở lại như trước khi đọc `E`.

- `E` là số: đẩy một giá trị, không đụng `ops`. ✔
- `E = op(A,B)`: đẩy `op`; theo quy nạp đọc `A` thêm giá trị `vA`, đọc `B` thêm `vB` (và `ops` vẫn có `op` ở đỉnh); gặp `)` lấy `vB`, `vA`, `op` ⇒ đẩy `op(vA, vB)` và `ops` trở lại như cũ. ✔

Chuỗi đầu vào là một biểu thức hoàn chỉnh ⇒ cuối cùng `vals = [giá trị]`.

**Cách 4:** ký pháp tiền tố với số ngôi cố định xác định duy nhất cây (đếm "số chỗ trống" từ trái sang luôn dương cho tới token cuối). Duyệt từ phải, mỗi phép toán gặp hai giá trị trên đỉnh ngăn xếp là hai cây con của nó theo đúng thứ tự trái (đỉnh) – phải.

**Chia:** `a // b` đúng khi `b | a`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thay thế lặp | `O(n · depth)` | `O(n)` | tới `O(n²)` |
| 2. Đệ quy | `O(n)` | `O(depth)` ngăn xếp gọi | lỗi khi `depth > ~1000` |
| 3. Hai ngăn xếp | `O(n)` | `O(depth)` | **an toàn** |
| 4. Ba Lan | `O(n)` | `O(n)` | gọn nhất |

(Phép nhân số lớn tốn thêm chi phí theo số chữ số, nhưng đề bảo đảm kết quả vừa 64 bit.)

## 12. Bộ test

In [12]:
METHODS = [brute, recursive, two_stacks, polish, lambda e: Solution().evaluateExpression(e)]
for e, exp in examples:
    assert all(f(e) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên
for e, exp in [("0", 0), ("-0", 0), ("123456789012", 123456789012), ("sub(-5,-7)", 2),
               ("div(-12,4)", -3), ("div(12,-4)", -3), ("mul(-3,-3)", 9), ("div(-7,-7)", 1)]:
    assert all(f(e) == exp for f in METHODS), e
print("Ca biên: OK")

Ca biên: OK


In [14]:
# 12.3 sinh biểu thức ngẫu nhiên (bảo đảm chia hết), so với giá trị tính trên cây
def gen(rng, depth):
    if depth == 0 or rng.random() < 0.3:
        v = rng.randint(-20, 20); return str(v), v
    op = rng.choice(["add", "sub", "mul", "div"])
    sa, a = gen(rng, depth - 1); sb, b = gen(rng, depth - 1)
    if op == "div":
        if b == 0:
            sb, b = "1", 1
        sa, a = f"mul({sa},{sb})", a * b           # a*b chia hết cho b
        return f"div({sa},{sb})", a // b
    return f"{op}({sa},{sb})", apply(op, a, b)

rng = random.Random(3749)
for _ in range(1500):
    e, v = gen(rng, rng.randint(0, 6))
    assert all(f(e) == v for f in METHODS), e
print("1500 biểu thức ngẫu nhiên: OK")

1500 biểu thức ngẫu nhiên: OK


In [15]:
# 12.4 biểu thức dài 10^5 ký tự, lồng rất sâu (đệ quy sẽ lỗi)
depth = 10**5 // 9
deep_expr = "add(" * depth + "1" + ",1)" * depth
t0 = time.time(); r = Solution().evaluateExpression(deep_expr); t1 = time.time()
print("độ dài:", len(deep_expr), "| kết quả:", r, "| khớp Ba Lan:", r == polish(deep_expr), "| thời gian:", round(t1 - t0, 3), "s")

độ dài: 77778 | kết quả: 11112 | khớp Ba Lan: True | thời gian: 0.009 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đệ quy với biểu thức lồng sâu

Như mục 7.1: `RecursionError` khi lồng quá ~1000 tầng. Tăng `sys.setrecursionlimit` có thể làm **sập** tiến trình (tràn stack C). Dùng ngăn xếp tường minh.

### Lỗi 2 — Dùng `float` để chia

In [16]:
def wrong_float_div(expr):
    ops, vals = [], []
    for t in TOKEN.findall(expr):
        if t in ("add", "sub", "mul", "div"): ops.append(t)
        elif t == ")":
            b = vals.pop(); a = vals.pop(); op = ops.pop()
            vals.append(int(a / b) if op == "div" else apply(op, a, b))     # SAI: float
        elif t not in "(,": vals.append(int(t))
    return vals[0]

e = "div(mul(3000000000000000001,3),3)"
print("sai:", wrong_float_div(e), "| đúng:", Solution().evaluateExpression(e))

sai: 3000000000000000000 | đúng: 3000000000000000001


### Lỗi 3 — Lấy hai giá trị ra khỏi ngăn xếp sai thứ tự

Đỉnh ngăn xếp là đối số **phải** (`b`) — `sub` và `div` không giao hoán.

In [17]:
def wrong_order(expr):
    ops, vals = [], []
    for t in TOKEN.findall(expr):
        if t in ("add", "sub", "mul", "div"): ops.append(t)
        elif t == ")":
            a = vals.pop(); b = vals.pop()                                  # SAI: đảo a, b
            vals.append(apply(ops.pop(), a, b))
        elif t not in "(,": vals.append(int(t))
    return vals[0]

print("sai:", wrong_order("sub(9,5)"), "| đúng:", Solution().evaluateExpression("sub(9,5)"))

sai: -4 | đúng: 4


### Lỗi 4 — Chỉ đọc một chữ số / bỏ quên dấu âm

In [18]:
def wrong_single_digit(expr):
    ops, vals = [], []
    i = 0
    while i < len(expr):
        ch = expr[i]
        if ch.isdigit(): vals.append(int(ch)); i += 1                       # SAI: một chữ số, bỏ dấu '-'
        elif ch.isalpha(): ops.append(expr[i:i + 3]); i += 4
        elif ch == ")":
            b = vals.pop(); a = vals.pop(); vals.append(apply(ops.pop(), a, b)); i += 1
        else: i += 1
    return vals[0]

try:
    print("sai:", wrong_single_digit("add(12,-3)"))
except Exception as ex:
    print("sai:", type(ex).__name__, "-", ex)
print("đúng:", Solution().evaluateExpression("add(12,-3)"))

sai: 1
đúng: 9


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ngữ pháp `op(E, E)` ⇒ cây biểu thức ⇒ đệ quy xuống **hoặc** ngăn xếp.
- Python + chuỗi `10^5` ⇒ cẩn thận đệ quy; ưu tiên ngăn xếp.
- Toán tử có số ngôi cố định ⇒ ngoặc là thừa ⇒ ký pháp Ba Lan.
- Đề nói "chia hết" ⇒ `//` an toàn; không bao giờ dùng `float` cho số nguyên lớn.

## 15. Biến thể của bài

1. **Thêm phép toán nhiều ngôi** (`add(a,b,c,...)`): cần đánh dấu bắt đầu danh sách đối số trên ngăn xếp.
2. **Chia làm tròn về 0** (như C++/Java) khi không chia hết: `int(a / b)` sai với số lớn ⇒ dùng `abs` và dấu.
3. **Biến** (`let(x, E, body)`): ngăn xếp phạm vi (LC 736 *Parse Lisp Expression*).
4. **Biểu thức trung tố** (`2+3*4`): thuật toán shunting-yard.

In [19]:
# Biến thể 1 + 2: phép toán nhiều ngôi và chia làm tròn về 0
def eval_nary(expr):
    stack = []                                   # chứa phép toán (str) hoặc giá trị (int)
    MARK = object()
    for t in TOKEN.findall(expr):
        if t in ("add", "sub", "mul", "div"):
            stack.append(t)
        elif t == "(":
            stack.append(MARK)
        elif t == ")":
            args = []
            while stack[-1] is not MARK:
                args.append(stack.pop())
            stack.pop(); op = stack.pop(); args.reverse()
            r = args[0]
            for x in args[1:]:
                if op == "add": r += x
                elif op == "sub": r -= x
                elif op == "mul": r *= x
                else:
                    q = abs(r) // abs(x); r = q if (r >= 0) == (x >= 0) else -q
            stack.append(r)
        elif t != ",":
            stack.append(int(t))
    return stack[0]

print(eval_nary("add(1,2,3,mul(2,2,2))"), eval_nary("div(-7,2)"), eval_nary("sub(10,1,2,3)"))
assert eval_nary("div(-7,2)") == -3 and eval_nary("add(1,2,3,mul(2,2,2))") == 14
for e, exp in examples:
    assert eval_nary(e) == exp
print("Biến thể OK")

14 -3 4
Biến thể OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | An toàn độ sâu | Khi nào dùng |
|---|---|---|---|---|
| 1. Thay thế lặp | Regex trong cùng | `O(n²)` | Có | Kiểm chứng |
| 2. Đệ quy | Recursive descent | `O(n)` | **Không** | Ngôn ngữ có stack lớn |
| 3. Hai ngăn xếp | Theo ngoặc | `O(n)` | Có | **Lời giải chuẩn** |
| 4. Ba Lan | Bỏ ngoặc, quét phải→trái | `O(n)` | Có | Gọn, đẹp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `"sub(mul(-2,3),div(8,-4))"` = ?

**Bài 2 (tính tay).** Viết dãy token Ba Lan của `"add(sub(1,2),mul(3,4))"` và tính.

**Bài 3 (điền TODO).** Viết `depth(expr)` trả về độ sâu cây biểu thức (số là độ sâu 0).

**Bài 4 (tự viết lại).** Viết hàm `to_infix(expr)` chuyển sang dạng trung tố có ngoặc đầy đủ, ví dụ `"add(2,3)" → "(2 + 3)"`.

**Bài 5 (mini challenge).** Viết bộ **kiểm tra hợp lệ** `is_valid(expr)` theo đúng ngữ pháp (không dùng `eval`).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def depth(expr):
    # TODO: đếm số '(' đang mở lớn nhất
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def is_valid(expr):
    # TODO: duyệt token, dùng bộ đếm "số biểu thức còn thiếu" hoặc ngăn xếp trạng thái
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `mul(-2,3) = −6`. (2) `div(8,−4) = −2`. (3) `−6 − (−2) = −4`.

**Bài 2.** (1) `[add, sub, 1, 2, mul, 3, 4]`. (2) Từ phải: `4, 3 → mul = 12`; `2, 1 → sub = −1`. (3) `add(−1, 12) = 11`.

**Bài 3.** (1) Độ sâu = số `(` đang mở tối đa. (2) Duyệt ký tự. (3) Trả max.

**Bài 4.** (1) Ngăn xếp chuỗi thay cho số. (2) Ánh xạ `add → +`, … (3) Ghép `"(" + a + " op " + b + ")"`.

**Bài 5.** (1) Trạng thái: "mong đợi biểu thức", "mong đợi `,`", "mong đợi `)`". (2) Ngăn xếp lưu đang ở đối số thứ mấy. (3) Cuối cùng ngăn xếp rỗng và đã đọc đúng một biểu thức.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().evaluateExpression("sub(mul(-2,3),div(8,-4))") == -4
assert [t for t in TOKEN.findall("add(sub(1,2),mul(3,4))") if t not in "(),"] == ["add", "sub", "1", "2", "mul", "3", "4"]
assert polish("add(sub(1,2),mul(3,4))") == 11
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def depth_sol(expr):
    d = best = 0
    for ch in expr:
        if ch == "(": d += 1; best = max(best, d)
        elif ch == ")": d -= 1
    return best

SYM = {"add": "+", "sub": "-", "mul": "*", "div": "/"}
def to_infix_sol(expr):
    ops, vals = [], []
    for t in TOKEN.findall(expr):
        if t in SYM: ops.append(t)
        elif t == ")":
            b = vals.pop(); a = vals.pop(); vals.append(f"({a} {SYM[ops.pop()]} {b})")
        elif t not in "(,": vals.append(t)
    return vals[0]

print(depth_sol(examples[2][0]), to_infix_sol(examples[2][0]))
rng = random.Random(4)
for _ in range(300):
    e, v = gen(rng, rng.randint(0, 5))
    inf = to_infix_sol(e).replace("/", "//")
    assert eval(inf) == v                      # chỉ dùng eval để KIỂM TRA bài tập (biểu thức tự sinh)
print("Bài 3, 4 OK")

3 ((4 * (9 - 5)) / (1 + 1))
Bài 3, 4 OK


In [24]:
# Bài 5: kiểm tra hợp lệ theo ngữ pháp
def is_valid_sol(expr):
    toks = TOKEN.findall(expr)
    if "".join(toks) != expr: return False             # có ký tự lạ
    pos = 0
    stack = []                                       # mỗi phần tử: số đối số đã đọc của phép toán đang mở
    need_expr = True; done = False
    while pos < len(toks):
        t = toks[pos]
        if done: return False
        if need_expr:
            if t in ("add", "sub", "mul", "div"):
                if pos + 1 >= len(toks) or toks[pos + 1] != "(": return False
                stack.append(0); pos += 2; continue
            if t in "(),": return False
            need_expr = False
        else:
            if not stack: return False
            if t == "," and stack[-1] == 0: stack[-1] = 1; need_expr = True
            elif t == ")" and stack[-1] == 1: stack.pop()
            else: return False
        if not need_expr and not stack and pos == len(toks) - 1: done = True
        pos += 1
    return not stack and not need_expr

tests = {"add(2,3)": True, "-42": True, examples[2][0]: True, "add(2)": False, "add(2,3,4)": False,
         "add(2,3))": False, "foo(1,2)": False, "add(,1)": False, "1,2": False, "": False}
for e, exp in tests.items():
    assert is_valid_sol(e) == exp, e
rng = random.Random(5)
assert all(is_valid_sol(gen(rng, 5)[0]) for _ in range(300))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ops, vals = [], []
for each token:
    số (có thể '-')  -> vals.append(int)
    tên phép toán    -> ops.append(op)    # bỏ qua '(' ngay sau
    ')'              -> b = vals.pop(); a = vals.pop(); vals.append(op(a, b))
    ','              -> bỏ qua
return vals[0]
```

**Checklist:**

- [ ] Không đệ quy sâu (chuỗi tới `10^5`)?
- [ ] `-` là dấu số âm, đọc số nhiều chữ số?
- [ ] Lấy `b` trước, `a` sau (đỉnh ngăn xếp là đối số phải)?
- [ ] Chia bằng `//` (đề bảo đảm chia hết), không dùng `float`?
- [ ] Biểu thức chỉ là một số cũng chạy đúng?